In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cpu
False


In [2]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')
    f.write('NA,Pave,127500\n')
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

In [3]:
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


##### 处理缺失值

In [4]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = inputs.fillna(inputs.mean(numeric_only=True))  #缺失值填充，填充为该列的均值
## 高版本pandas的numeric_only参数需要指定参数值，否则默认是False，例子：inputs = inputs.fillna(inputs.mean(numeric_only=True))
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


In [5]:
inputs = pd.get_dummies(inputs, dummy_na=True, dtype=int)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0           1          0
1       2.0           0          1
2       4.0           0          1
3       3.0           0          1


##### 转换为张量格式

In [6]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

### 作业

In [7]:
#创建原始数据集
import os 

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_base = os.path.join('..', 'data', 'NBA Player scoring data')
with open(data_base, 'w', encoding='utf-8') as t:
    t.write("球员,年龄,比赛场次,得分\n")
    t.write("东契奇,NA,61,33.6\n")
    t.write("亚历山大,28,NA,NA\n")
    t.write("爱德华兹,25,NA,29.5\n")
    t.write("马克西,26,NA,29\n")
    t.write("布朗,NA,NA,28.6\n")
    t.write("莱昂纳德,NA,57,28.3\n")

In [8]:
import pandas as pd

df = pd.read_csv(data_base)
print(df)

     球员    年龄  比赛场次    得分
0   东契奇   NaN  61.0  33.6
1  亚历山大  28.0   NaN   NaN
2  爱德华兹  25.0   NaN  29.5
3   马克西  26.0   NaN  29.0
4    布朗   NaN   NaN  28.6
5  莱昂纳德   NaN  57.0  28.3


In [9]:
#删除缺失值最多的列
df = df.drop(columns=df.isna().sum(axis=0).idxmax())
print(df)

     球员    年龄    得分
0   东契奇   NaN  33.6
1  亚历山大  28.0   NaN
2  爱德华兹  25.0  29.5
3   马克西  26.0  29.0
4    布朗   NaN  28.6
5  莱昂纳德   NaN  28.3


In [10]:
inp, outp = df.iloc[:, 1], df.iloc[:, 2]
inp = inp.fillna(inp.mean())
outp = outp.fillna(outp.mean())
print(inp, outp)
print(df)

0    26.333333
1    28.000000
2    25.000000
3    26.000000
4    26.333333
5    26.333333
Name: 年龄, dtype: float64 0    33.6
1    29.8
2    29.5
3    29.0
4    28.6
5    28.3
Name: 得分, dtype: float64
     球员    年龄    得分
0   东契奇   NaN  33.6
1  亚历山大  28.0   NaN
2  爱德华兹  25.0  29.5
3   马克西  26.0  29.0
4    布朗   NaN  28.6
5  莱昂纳德   NaN  28.3


In [11]:
import torch 

a = torch.tensor(inp.to_numpy(dtype=float))
b = torch.tensor(outp.to_numpy(dtype=float))
a, b

(tensor([26.3333, 28.0000, 25.0000, 26.0000, 26.3333, 26.3333],
        dtype=torch.float64),
 tensor([33.6000, 29.8000, 29.5000, 29.0000, 28.6000, 28.3000],
        dtype=torch.float64))